# Module 4 — Matching, incl. Propensity Score Matching

Part of the Causal Inference Playbook. Program and progress: [CURRICULUM.md](CURRICULUM.md).

## Why you're here: what this module is for

- The running question: did NSW (National Supported Work), a US job-training program for men with very low earnings, raise the earnings of the men who took it?
- The men who took the program are called the treated men. They're compared with men drawn from the PSID (Panel Study of Income Dynamics), a US household survey, who didn't take it: the comparison men. Each man has "before" columns: facts about him from before the program, such as his age, his education and his earlier earnings. The outcome is his earnings after the program.
- The treated men were different from the comparison men even before the program started: for example, they earned far less. So a plain comparison of the two groups mixes the program's effect with those differences.
- Module 4 uses matching: for each treated man, find the comparison man who looked most like him before the program, and compare the two. Only the comparison men chosen as a match take part in the estimate.
- What can go wrong: a treated man with nobody like him still gets the closest comparison man available, however different that man is.

**What this module produces:**
1. A matching estimate of the program's effect on the real data, a check of whether the matches look like the treated men, and the estimate's uncertainty.
2. A comparison with regression adjustment, which predicts each treated man's earnings without the program from a formula fitted on the comparison men, and the reason the two differ.

## From the curriculum

> ## <u>Module 4 — Matching, incl. Propensity Score Matching</u>
>
> **Concepts**
> - Matching: for each treated man (a man who took the program in 1976–77),
>   find his match: the comparison man who looked most like him before the
>   program. His match's 1978 earnings stand in for his earnings without the
>   program.
> - The propensity score: among all the men, treated and comparison, with
>   "before" columns like his, the share who were treated. It's computed from
>   his columns alone, not from which group he's actually in. Matching on this one number instead of on all
>   the columns is propensity score matching.
> - Balance check: after matching, compare the mean of every "before" column
>   for the treated men and for their matches.
> - Overlap: matching only works for treated men who have comparison men with
>   a similar score.
>
> **Why it matters**
> Matching uses only the comparison men chosen as matches, and the analyst can
> see each match. So it shows plainly how alike the groups were made, and which
> treated men had nobody like them.
>
> **Worked example**
> 1. NSW (National Supported Work) job-training data vs the PSID (Panel Study
>    of Income Dynamics) men: the score, the matches, the balance check and
>    the effect estimate, with a 95% confidence interval.
> 2. Matching vs regression adjustment on the same data, and why they
>    differ.
>
> **Resources**
> - [Causal Inference: The Mixtape, ch. 5 "Unconfoundedness"](https://mixtape.scunning.com/05-causal_cross_sectional_designs)
>   (Scott Cunningham, ~60 min for sections 5.5–5.8, verified): exact
>   matching, nearest-neighbour matching and the propensity score.
> - [The Effect, ch. 14 "Matching"](https://theeffectbook.net/ch-Matching.html)
>   (Nick Huntington-Klein, ~60 min, verified): matching, the propensity
>   score, overlap and balance checks.
>
> **Exercise type:** redo the propensity score matching with the CPS (Current
> Population Survey, a monthly US labour survey) comparison group, and check
> the matches. Fully worked.
>
> **Interview angle:** "We matched users who used feature X to similar users
> who didn't, using a propensity score. The matched users with X spend 10%
> more. Is that the effect of X?"

## Lesson

### The data

- The treated men: 185 men who took NSW in 1976 or 1977. They had applied, and a lottery let them in.
- The comparison men: 2,490 men drawn from the PSID, who didn't apply to the program, so didn't take it.
- The 8 "before" columns: age, years of education, black, hispanic, married, no degree (no high-school degree), 1974 earnings and 1975 earnings. Black, hispanic, married and no degree are yes/no columns: 1 for yes, 0 for no.
- The outcome: 1978 earnings. All earnings are in US dollars per year.
- The benchmark: the same lottery kept 260 applicants out. The lottery, not the men, decided who got in, so the 185 treated men and these 260 men differ only by chance. So the 260 men's mean 1978 earnings shows what the treated men would have earned without the program. The treated men's mean minus the 260 men's mean is the program's effect on the treated men, with no selection bias (the part of a gap that comes from the groups being different to begin with). Every estimate below is compared with it.
- The naive gap: the plain comparison of the treated men with the PSID men. It mixes the program's effect with how different the two groups were.

**The motive:** the benchmark, the same one used in Modules 1–3, is the answer each method below should get close to. The naive gap is the unadjusted comparison each method tries to improve on.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

import common
from common import show

exp = common.load_lalonde("experimental")  # the lottery: 185 treated men (treat = 1) and the 260 men kept out (treat = 0)
psid = common.load_lalonde("psid")  # the same 185 treated men, plus the 2,490 PSID men (treat = 0)
nsw_t = exp[exp.treat == 1].copy()
lottery = exp[exp.treat == 0]
psid_c = psid[psid.treat == 0].copy()
COLS8 = ["age", "education", "black", "hispanic", "married", "nodegree", "re74", "re75"]

benchmark = nsw_t.re78.mean() - lottery.re78.mean()
print(f"Benchmark = (mean 1978 earnings of the treated men, {len(nsw_t)} men) - (mean 1978 earnings of the men kept out by the lottery, {len(lottery)} men) = "
      f"{nsw_t.re78.mean():,.1f} - {lottery.re78.mean():,.1f} = {benchmark:,.1f}")
naive = nsw_t.re78.mean() - psid_c.re78.mean()
print(f"Naive gap = (mean 1978 earnings of the treated men, {len(nsw_t)} men) - (mean 1978 earnings of the PSID men, {len(psid_c):,} men) = "
      f"{nsw_t.re78.mean():,.1f} - {psid_c.re78.mean():,.1f} = {naive:,.1f}")

Benchmark = (mean 1978 earnings of the treated men, 185 men) - (mean 1978 earnings of the men kept out by the lottery, 260 men) = 6,349.1 - 4,554.8 = 1,794.3
Naive gap = (mean 1978 earnings of the treated men, 185 men) - (mean 1978 earnings of the PSID men, 2,490 men) = 6,349.1 - 21,553.9 = -15,204.8


**Observation:** the naive gap, (−$15,204.8), makes the program look harmful. The benchmark, $1,794.3, says it raised earnings.

### Matching on all 8 columns at once?

The simplest matching would pair each treated man with a PSID man exactly the same on all 8 columns. The count below shows how many treated men have one. It also counts how many of those earned $0 in both 1974 and 1975: two men who both earned nothing match easily on earnings.

In [2]:
psid_rows = set(map(tuple, psid_c[COLS8].to_numpy()))
has_exact = np.array([tuple(row) in psid_rows for row in nsw_t[COLS8].to_numpy()])
exact = nsw_t[has_exact]
print(f"Treated men with a PSID man exactly the same on all 8 columns = {has_exact.sum()} of {len(nsw_t)}")
print(f"Of these, men who earned $0 in both 1974 and 1975 = {((exact.re74 == 0) & (exact.re75 == 0)).sum()} of {len(exact)}")

Treated men with a PSID man exactly the same on all 8 columns = 12 of 185
Of these, men who earned $0 in both 1974 and 1975 = 12 of 12


**Observation:** only 12 of the 185 treated men have an exact match, and all 12 earned $0 in both 1974 and 1975. The earnings columns are in dollars and cents, so two men who earned anything rarely earned exactly the same.

**Conclusion:** exact matching works for only 12 of the 185 treated men, so it can't be used here.

Matching on the closest man instead of an exact one needs a rule for "closest" across 8 columns: is 1 year of age closer than $1,000 of earnings? The propensity score answers this by turning the 8 columns into one number, so that "closest" means closest score.

### The propensity score

- A man's propensity score is the share of treated men among the men whose 8 "before" columns are like his, out of all 2,675 men: the 185 treated men and the 2,490 PSID men. It's computed from his columns alone, not from which group he's actually in, so a PSID man can have a high score. It's a number between 0 and 1.
- It's found by logistic regression: a regression that predicts a yes/no column (here, 1 for a treated man, 0 for a PSID man) from other columns. Unlike a straight line, its predictions never go below 0 or above 1, so they can be read as chances.
- The score is the rule for "closest": two men are close if their scores are close.

Why matching on one number is enough to make the groups alike: because the score is fitted to tell treated men from PSID men using the 8 columns. If, among men with the same score, some column still told the two groups apart, the fit would have used that column to give them different scores. So the treated men and their matches should end up alike, on average, on each of the 8 columns.

### Matching in three steps

The question is about the treated men: the ATT (average treatment effect on the treated) is how much the program changed their 1978 earnings, on average. For each treated man, the missing number is what he would have earned without the program.

1. For each treated man, find his match: the PSID man whose score is closest to his. If several PSID men tie for closest, all of them are his match.
2. His match's 1978 earnings stand in for his 1978 earnings without the program. With several tied matches, their mean stands in.
3. ATT estimate = the mean of (his actual 1978 earnings − his match's 1978 earnings), over the treated men.

The same PSID man can be the match of several treated men, so a few PSID men can make up many of the matches.

The score makes the treated men and their matches alike only on average, not for each man and his match, and only if the score's formula fits the data well. So this has to be checked on the data, column by column. The balance check compares the mean of each "before" column for the treated men, for all the PSID men and for the matches. For each column, the difference = (a group's mean) − (the treated men's mean), once for all the PSID men and once for the matches. Matching worked if the differences left for the matches are small.

### What it assumes

1. Every confounder has a column. A confounder is a trait that affects both who took the program and earnings afterwards. Matching can only make the men alike on the columns it sees. A confounder with no column (motivation, for example) isn't matched on.
2. Overlap: every treated man has PSID men with a similar score. A treated man with a higher score than every PSID man still gets the closest one, and that man can be quite different from him.

## Worked example 1: propensity score matching on the real data

**The motive:** the naive gap, (−$15,204.8), says the program cut earnings by over $15,000, against a benchmark of $1,794.3. If matching on the score works, the estimate should move toward the benchmark.

### Step 1 of 4: each man's propensity score

**The motive:** matching needs a PSID man with a similar score for each treated man. The table counts the men in each score range, to show where such PSID men exist. The score is fitted by logistic regression on all 2,675 men: the 185 treated men and the 2,490 PSID men.

In [3]:
SCORE = "treat ~ " + " + ".join(COLS8)
both = pd.concat([nsw_t, psid_c], ignore_index=True)
both["score"] = smf.logit(SCORE, both).fit(disp=0).predict(both)
nsw_t["score"] = both.score[both.treat == 1].to_numpy()
psid_c["score"] = both.score[both.treat == 0].to_numpy()

edges = np.linspace(0, 1, 11)
labels = [f"{a:.1f} < score <= {b:.1f}" for a, b in zip(edges[:-1], edges[1:])]
show(pd.DataFrame({
    "propensity score": labels,
    f"treated men ({len(nsw_t)} men)": pd.cut(nsw_t.score, edges, labels=labels).value_counts().reindex(labels).to_numpy(),
    f"PSID men ({len(psid_c):,} men)": pd.cut(psid_c.score, edges, labels=labels).value_counts().reindex(labels).to_numpy(),
}))

propensity score    treated men (185 men)  PSID men (2,490 men)
0.0 < score <= 0.1                     10                  2326
0.1 < score <= 0.2                      8                    72
0.2 < score <= 0.3                     13                    28
0.3 < score <= 0.4                     11                    20
0.4 < score <= 0.5                     13                    11
0.5 < score <= 0.6                     15                     8
0.6 < score <= 0.7                     11                     7
0.7 < score <= 0.8                     28                     7
0.8 < score <= 0.9                     47                     8
0.9 < score <= 1.0                     29                     3


**Observation:**
- 2,326 of the 2,490 PSID men have a score of 0.1 or less: by their 8 columns, each has at most a 10% chance of being a treated man. Only 10 treated men have a score that low.
- Above 0.7, there are 28 + 47 + 29 = 104 treated men and only 7 + 8 + 3 = 18 PSID men.

**Conclusion:** most PSID men have scores where almost no treated man's is, so few of them can be anyone's close match. At high scores, matches are scarce: 18 PSID men for 104 treated men.

### Step 2 of 4: match each treated man with the PSID man with the closest score

**The motive:** this step finds each treated man's match. The table shows the first 5 treated men in the data and their matches, with 1975 earnings as one example column. If a treated man has several tied matches, the table shows their mean. The lines below the table show how often the same PSID men are reused, and how many treated men have no PSID man with a score as high as theirs.

In [4]:
def match(treated, comparison):
    # One row per treated man, one column per comparison man.
    # A cell is 1 if that comparison man is his match, 1 / k each if k comparison men tie for closest, else 0.
    distance = np.abs(treated.score.to_numpy()[:, None] - comparison.score.to_numpy()[None, :])
    closest = distance <= distance.min(axis=1, keepdims=True) + 1e-12  # the tiny allowance counts scores equal up to rounding as tied
    return closest / closest.sum(axis=1, keepdims=True)

w = match(nsw_t, psid_c)
match_mean = lambda col: w @ psid_c[col].to_numpy()  # each treated man's match value (the mean, if several tie)

show(pd.DataFrame({
    "treated man": [f"treated man {i + 1}" for i in range(5)],
    "score": [f"{v:.3f}" for v in nsw_t.score[:5]],
    "1975 earnings": [f"{v:,.1f}" for v in nsw_t.re75[:5]],
    "1978 earnings": [f"{v:,.1f}" for v in nsw_t.re78[:5]],
    "his match: score": [f"{v:.3f}" for v in match_mean("score")[:5]],
    "his match: 1975 earnings": [f"{v:,.1f}" for v in match_mean("re75")[:5]],
    "his match: 1978 earnings": [f"{v:,.1f}" for v in match_mean("re78")[:5]],
}))
print()

times_used = (w > 0).sum(axis=0)
top = psid_c.iloc[times_used.argmax()]
above_all = (nsw_t.score > psid_c.score.max()).sum()
print(f"PSID men used as a match = {(times_used > 0).sum()} of {len(psid_c):,}")
print(f"Treated men with a higher score than every PSID man = {above_all} of {len(nsw_t)}")
print(f"Treated men with a lower score than every PSID man = {(nsw_t.score < psid_c.score.min()).sum()} of {len(nsw_t)}")
print(f"Most-used PSID man: the match of {times_used.max()} treated men (score {top.score:.3f}, the highest of any PSID man)")
on_top = w[:, times_used.argmax()] > 0
print(f"   of these {times_used.max()} treated men, with a higher score than every PSID man = {(on_top & (nsw_t.score.to_numpy() > top.score)).sum()}, "
      f"with a score below his = {(on_top & (nsw_t.score.to_numpy() <= top.score)).sum()}")

treated man    score  1975 earnings  1978 earnings  his match: score  his match: 1975 earnings  his match: 1978 earnings
treated man 1  0.377            0.0        9,930.0             0.380                       0.0                       0.0
treated man 2  0.885            0.0        3,595.9             0.885                     895.2                   2,305.3
treated man 3  0.720            0.0       24,909.4             0.722                   2,685.5                   8,127.5
treated man 4  0.872            0.0        7,506.1             0.877                       0.0                       0.0
treated man 5  0.790            0.0          289.8             0.791                       0.0                  10,344.1

PSID men used as a match = 72 of 2,490
Treated men with a higher score than every PSID man = 23 of 185
Treated men with a lower score than every PSID man = 0 of 185
Most-used PSID man: the match of 26 treated men (score 0.915, the highest of any PSID man)
   of these 26 tr

**Observation:**
- Each match has almost the same score as its treated man, but not always the same values on each column: treated man 3 earned $0 in 1975, and his match earned $2,685.5.
- A treated man and his match can differ a lot in 1978 earnings: treated man 3 earned $24,909.4, his match $8,127.5. The estimate is the mean over all 185 treated men, not any one man.
- Only 72 of the 2,490 PSID men are used as a match. The other 2,490 − 72 = 2,418 play no part in the estimate.
- 23 treated men have a higher score than every PSID man, so their closest PSID man is the one with the highest score. He's also the closest for 3 treated men with a score below his: 26 in all. The 23 men above him have nobody with a score as high as theirs: the overlap problem from the lesson.

**Conclusion:** a few PSID men make up many of the matches, and 23 treated men are matched with a man less likely than them to be treated.

### Step 3 of 4: the balance check

**The motive:** matching was done on the score alone. If the score did its job, the treated men and their matches should be alike, on average, on all 8 columns. The table compares the means, next to all the PSID men before matching. For a 1/0 column, the mean is the share of men with a 1: for example, the share who are black. The last two columns show each difference, before and after matching: difference before = (all PSID men's mean) − (the treated men's mean), and difference after = (the matches' mean) − (the treated men's mean). For a 1/0 column, the difference is in percentage points.

In [5]:
names = {"age": "age", "education": "years of education", "black": "black", "hispanic": "hispanic", "married": "married",
         "nodegree": "no degree", "re74": "1974 earnings", "re75": "1975 earnings"}
yes_no = {"black", "hispanic", "married", "nodegree"}
fmt = lambda col, v: f"{v:.1%}" if col in yes_no else f"{v:,.1f}"
gap = lambda col, v: f"{v * 100 + 0:,.1f}" if col in yes_no else f"{v + 0:,.1f}"  # differences in 1/0 columns, in percentage points

shown = lambda col, v: round(v, 3) if col in yes_no else round(v, 1)  # differences use the means as printed, so the subtraction checks by hand

def balance(treated, comparison, w, label):
    return pd.DataFrame({
        "column": [names[c] for c in COLS8],
        "type": ["1/0" if c in yes_no else "number" for c in COLS8],
        f"treated men ({len(treated)} men): mean": [fmt(c, treated[c].mean()) for c in COLS8],
        f"all {label} men ({len(comparison):,} men): mean": [fmt(c, comparison[c].mean()) for c in COLS8],
        f"their matches ({len(treated)} matches, from {((w > 0).sum(axis=0) > 0).sum()} {label} men): mean": [fmt(c, (w @ comparison[c].to_numpy()).mean()) for c in COLS8],
        "difference before matching": [gap(c, shown(c, comparison[c].mean()) - shown(c, treated[c].mean())) for c in COLS8],
        "difference after matching": [gap(c, shown(c, (w @ comparison[c].to_numpy()).mean()) - shown(c, treated[c].mean())) for c in COLS8],
    })

show(balance(nsw_t, psid_c, w, "PSID"))

column              type    treated men (185 men): mean  all PSID men (2,490 men): mean  their matches (185 matches, from 72 PSID men): mean  difference before matching  difference after matching
age                 number                         25.8                            34.9                                                 23.5                         9.1                       -2.3
years of education  number                         10.3                            12.1                                                 10.4                         1.8                        0.1
black               1/0                           84.3%                           25.1%                                                82.2%                       -59.2                       -2.1
hispanic            1/0                            5.9%                            3.3%                                                 7.0%                        -2.6                        1.1
married             

**Observation:**
- Every difference shrank in size, ignoring its sign: for example, 1975 earnings from $17,531.2 to $730.1.
- Differences are left on every column, for example 1974 earnings ($1,160.4), married ((−9.2) points) and age ((−2.3) years).

**The easy test:** every difference shrank, so matching passes. It's easy to check, but it says nothing about how big the differences left are. A difference can shrink and still be large: this test can't say whether $1,160.4 of 1974 earnings is a lot.

### The size test

**The motive:** to judge whether a difference is big, compare it with how much the column varies among the treated men. A difference of 2.3 years of age is large if most treated men's ages are close to each other, and small if they spread over decades.
- Spread = the column's standard deviation among the treated men: the typical distance of a treated man's value from the treated men's mean.
- Size = (difference after matching) ÷ (spread). It has no unit, so dollars, years and percentage points can be compared.
- The rule: a column passes if its size, ignoring the sign, is under 0.1. This is the most common balance test.

In [6]:
def size_test(treated, comparison, w):
    rows = []
    for c in COLS8:
        to_points = 100 if c in yes_no else 1  # 1/0 columns in percentage points
        after = round((shown(c, (w @ comparison[c].to_numpy()).mean()) - shown(c, treated[c].mean())) * to_points, 1)
        spread = round(treated[c].std() * to_points, 1)
        size = after / spread
        rows.append((names[c], "1/0" if c in yes_no else "number", f"{after + 0:,.1f}", f"{spread:,.1f}", f"{size + 0:.3f}", "yes" if abs(size) < 0.1 else "no"))
    return pd.DataFrame(rows, columns=["column", "type", "difference after matching", f"spread among the treated men ({len(treated)} men)",
                                       "size = difference ÷ spread", "passes (size under 0.1)?"])

show(size_test(nsw_t, psid_c, w))

column              type    difference after matching  spread among the treated men (185 men)  size = difference ÷ spread  passes (size under 0.1)?
age                 number                       -2.3                                     7.2                      -0.319  no
years of education  number                        0.1                                     2.0                       0.050  yes
black               1/0                          -2.1                                    36.5                      -0.058  yes
hispanic            1/0                           1.1                                    23.7                       0.046  yes
married             1/0                          -9.2                                    39.3                      -0.234  no
no degree           1/0                          -2.2                                    45.6                      -0.048  yes
1974 earnings       number                    1,160.4                                 4,886.

**Observation:** 4 of the 8 columns fail: age (−0.319), married (−0.234), 1974 earnings (0.237) and 1975 earnings (0.227). The other 4 pass.

**Conclusion:** by the size test, the most common balance test, matching on the PSID men fails: the opposite of the easy test. The matches are younger, less often married, and earned more in 1974 and 1975 than the treated men. These differences may still be mixed into the estimate in step 4.

### Step 4 of 4: the ATT estimate, and how sure it is

**The motive:** with the matches set, the ATT estimate is the mean of (actual − match) over the treated men. Then the 95% confidence interval shows how much it could move if the study were rerun on another sample of treated men and PSID men.

The interval comes from the standard error, found with the standard formula for matching (from the economists Abadie and Imbens). The formula is long, so the code computes it. It has two parts. The first is the usual uncertainty of a mean, here the mean of (actual − match). The second is extra: each PSID man's 1978 earnings are partly chance, since a man like him could have earned more or less. A man used as a match 26 times brings that chance into the estimate 26 times.

The 95% confidence interval = ATT estimate ± (1.96 × standard error).

In [7]:
def matching_att(treated, comparison):
    # Returns the ATT estimate and the two parts of its standard error (the Abadie-Imbens formula for matching).
    # Simplification: each man's score is used as if known exactly, not as an estimate itself.
    w = match(treated, comparison)
    y_t, y_c = treated.re78.to_numpy(), comparison.re78.to_numpy()
    diff = y_t - w @ y_c
    att = diff.mean()
    # Each comparison man's chance part: how far his 1978 earnings sit from those of the comparison man (or men) with the closest score.
    distance = np.abs(comparison.score.to_numpy()[:, None] - comparison.score.to_numpy()[None, :])
    np.fill_diagonal(distance, np.inf)
    closest = distance <= distance.min(axis=1, keepdims=True) + 1e-12
    n_close = closest.sum(axis=1)
    chance = n_close / (n_close + 1) * (y_c - (closest / n_close[:, None]) @ y_c) ** 2
    used, used_sq = w.sum(axis=0), (w ** 2).sum(axis=0)
    n = len(treated)
    spread = ((diff - att) ** 2).sum() / n ** 2  # the usual uncertainty of a mean of the differences
    reuse = ((used ** 2 - used_sq) * chance).sum() / n ** 2  # extra: a man used k times brings his chance part in k times
    return att, np.sqrt(spread), np.sqrt(reuse)

att_match, se_mean, se_reuse = matching_att(nsw_t, psid_c)
# Every printed result below is computed from the printed (rounded) numbers, so each line checks by hand.
r1 = lambda v: round(float(v), 1)
se = r1(np.sqrt(r1(se_mean) ** 2 + r1(se_reuse) ** 2))
margin = r1(1.96 * se)
print(f"ATT estimate = (mean actual 1978 earnings, treated men, {len(nsw_t)} men) - (mean 1978 earnings of their matches, {len(nsw_t)} matches) = "
      f"{nsw_t.re78.mean():,.1f} - {match_mean('re78').mean():,.1f} = {att_match:,.1f}")
print()
print(f"Standard error = square root of ((usual uncertainty of the mean)² + (extra from reused matches)²) = square root of ({se_mean:,.1f}² + {se_reuse:,.1f}²) = {se:,.1f}")
print(f"95% confidence interval = {att_match:,.1f} ± (1.96 × {se:,.1f}) = {att_match:,.1f} ± {margin:,.1f} = {r1(att_match) - margin:,.1f} to {r1(att_match) + margin:,.1f}")

ATT estimate = (mean actual 1978 earnings, treated men, 185 men) - (mean 1978 earnings of their matches, 185 matches) = 6,349.1 - 4,223.4 = 2,125.7

Standard error = square root of ((usual uncertainty of the mean)² + (extra from reused matches)²) = square root of (660.6² + 682.2²) = 949.6
95% confidence interval = 2,125.7 ± (1.96 × 949.6) = 2,125.7 ± 1,861.2 = 264.5 to 3,986.9


**Observation:**
- Matching moves the estimate from the naive gap, (−$15,204.8), to $2,125.7. Gap to the benchmark = 1,794.3 − 2,125.7 = (−$331.4): the estimate is $331.4 above it.
- The 95% confidence interval runs from $264.5 to $3,986.9. It doesn't contain 0: the estimate rules out, at the 95% level, that the program did nothing.

**Conclusion:** the naive gap missed the benchmark by 1,794.3 − (−15,204.8) = $16,999.1. Propensity score matching misses it by $331.4, about a third of its standard error ($949.6). A miss that size is well within what chance alone could produce for this estimate.

## Worked example 2: matching vs regression adjustment

Regression adjustment, taught in Module 3, is the other common way to adjust for the "before" columns:
1. Fit a straight-line formula on the PSID men: predicted 1978 earnings = (starting point) + (a coefficient × each of the 8 columns).
2. Use it to predict each treated man's 1978 earnings without the program.
3. ATT estimate = the mean of (actual − predicted), over the treated men.

**The motive:** both methods adjust for the same 8 columns. If they give different answers, the reason shows what each method does differently.

In [8]:
LINEAR = "re78 ~ " + " + ".join(COLS8)
def regression_att(comparison):
    # Regression adjustment: fit on the comparison men, predict the treated men without the program, average actual - predicted.
    return (nsw_t.re78 - smf.ols(LINEAR, comparison).fit().predict(nsw_t)).mean()

att_reg = regression_att(psid_c)
print(f"Regression adjustment, fitted on all the PSID men ({len(psid_c):,} men): ATT estimate = {att_reg:,.1f}")

Regression adjustment, fitted on all the PSID men (2,490 men): ATT estimate = 790.5


**Observation:** regression adjustment gives $790.5. Gap to the benchmark = 1,794.3 − 790.5 = $1,003.8, three times matching's miss of $331.4.

**Hypothesis:** regression fits its one formula to all 2,490 PSID men, including the many who are unlike every treated man. Matching never uses those men. If that's the difference, regression fitted without them should land much closer to the benchmark.

**The test:** keep only the PSID men with a score at least as high as the lowest treated man's, and refit the regression on them. This drops only the PSID men with a lower score than every treated man's. There's no need for a cut at the top: no PSID man has a higher score than every treated man (step 2 of worked example 1). It doesn't pick the closest ones, as matching does.

Regression's intervals come from the bootstrap, as in Module 3: draw the men again at random, with replacement, 1,000 times, redo the regression each time, and keep the middle 95% of the estimates. Matching's interval is the one from step 4.

In [9]:
in_range = psid_c[psid_c.score >= nsw_t.score.min()]
print(f"Lowest score of any treated man = {nsw_t.score.min():.4f}")
print(f"PSID men with a lower score than every treated man = {len(psid_c) - len(in_range):,} of {len(psid_c):,}")
print()
def regression_interval(comparison):
    # Bootstrap: redraw the treated men and the comparison men with replacement, redo the regression, keep the middle 95%.
    add_one = lambda d: np.column_stack([np.ones(len(d)), d[COLS8].to_numpy(float)])
    X_c, y_c, X_t, y_t = add_one(comparison), comparison.re78.to_numpy(float), add_one(nsw_t), nsw_t.re78.to_numpy(float)
    rng = np.random.default_rng(3)
    boot = []
    for _ in range(1000):
        i_c, i_t = rng.integers(0, len(y_c), len(y_c)), rng.integers(0, len(y_t), len(y_t))
        coef = np.linalg.lstsq(X_c[i_c], y_c[i_c], rcond=None)[0]
        boot.append((y_t[i_t] - X_t[i_t] @ coef).mean())
    low, high = np.percentile(boot, [2.5, 97.5])
    return f"{low:,.1f} to {high:,.1f}"

rows = [
    ("regression adjustment, fitted on all the PSID men", f"{len(psid_c):,}", att_reg, regression_interval(psid_c)),
    ("regression adjustment, fitted only on the PSID men with a score at least the lowest treated man's", f"{len(in_range):,}", regression_att(in_range), regression_interval(in_range)),
    ("propensity score matching", f"{(times_used > 0).sum()}", att_match, f"{r1(att_match) - margin:,.1f} to {r1(att_match) + margin:,.1f}"),
]
show(pd.DataFrame({
    "method": [r[0] for r in rows],
    "PSID men used (to fit the formula, or as matches)": [r[1] for r in rows],
    "ATT estimate": [f"{r[2]:,.1f}" for r in rows],
    "95% confidence interval": [r[3] for r in rows],
    f"gap to the benchmark ({benchmark:,.1f} - ATT estimate)": [f"{r1(benchmark) - r1(r[2]):,.1f}" for r in rows],
}))

Lowest score of any treated man = 0.0002
PSID men with a lower score than every treated man = 1,330 of 2,490



method                                                                                             PSID men used (to fit the formula, or as matches)  ATT estimate  95% confidence interval  gap to the benchmark (1,794.3 - ATT estimate)
regression adjustment, fitted on all the PSID men                                                                                              2,490         790.5  -693.0 to 2,340.1                                              1,003.8
regression adjustment, fitted only on the PSID men with a score at least the lowest treated man's                                              1,160       1,931.2  203.8 to 3,609.2                                                -136.9
propensity score matching                                                                                                                         72       2,125.7  264.5 to 3,986.9                                                -331.4


**Observation:**
- 1,330 of the 2,490 PSID men have a lower score than every treated man.
- Without them, regression adjustment moves from $790.5 to $1,931.2. Its gap to the benchmark shrinks from $1,003.8 to (−$136.9): a negative gap means the estimate is above the benchmark.

**Conclusion:** the hypothesis holds. Most of regression's miss came from the 1,330 PSID men with a lower score than every treated man's. Because regression picks the coefficients that fit all the men it's given, those 1,330 men moved the coefficients, and the coefficients set the treated men's predictions. Without them, regression misses the benchmark by $136.9, and matching by $331.4. The trimmed regression's interval, like matching's, doesn't contain 0. Regression on all the PSID men can't rule out 0. The two estimates are $194.5 apart, while each interval is over $3,400 wide, so this data can't say which method is closer. What helped here is leaving out the comparison men unlike any treated man, which matching does by design.

## Exercise — propensity score matching with the CPS men

Redo the matching with a different comparison group: the 15,992 CPS (Current Population Survey, a monthly US labour survey) men, instead of the PSID men. Fit the scores with the same 8 columns, and match each treated man with the CPS man (or men, if several tie) with the closest score.

Write your answers first, then read the answers below.

### Questions

1. What is the ATT estimate, its 95% confidence interval, and its gap to the benchmark, $1,794.3?
2. Overlap and reuse: how many treated men have a higher score than every CPS man, and how many a lower one? How many CPS men are used as matches, and how many treated men does the most-used one match?
3. The balance check: for each of the 8 columns, what is the difference to the treated men's mean before and after matching? Does matching pass the easy test (every difference shrank)? Does it pass the size test?

### Answers

**Answer 1.**
- ATT estimate = (mean actual 1978 earnings, treated men, 185 men) − (mean 1978 earnings of their matches, 185 matches) = 6,349.1 − 4,555.2 = 1,793.9.
- Standard error = square root of ((usual uncertainty of the mean)² + (extra from reused matches)²) = square root of (669.1² + 383.3²) = 771.1.
- 95% confidence interval = 1,793.9 ± (1.96 × 771.1) = 1,793.9 ± 1,511.4 = 282.5 to 3,305.3.
- Gap to the benchmark = 1,794.3 − 1,793.9 = 0.4.
- Being $0.4 from the benchmark is luck, not precision: the interval spans 3,305.3 − 282.5 = $3,022.8.

**Answer 2.**
- 0 treated men have a higher score than every CPS man, and 0 a lower one. With the PSID men: 23 and 0.
- 194 CPS men are used as matches (194 / 15,992 = 1.2%): with ties, a treated man can have several. The most-used one is the match of 6 treated men. With the PSID men, it was 72 men (72 / 2,490 = 2.9%), and one was the match of 26.

**Answer 3.** It fails the easy test: one difference grew.

| column | type | treated men (185 men): mean | all CPS men (15,992 men): mean | their matches (185 matches, from 194 CPS men): mean | difference before matching | difference after matching |
|---|---|---|---|---|---|---|
| age | number | 25.8 | 33.2 | 23.0 | 7.4 | −2.8 |
| years of education | number | 10.3 | 12.0 | 10.3 | 1.7 | 0.0 (rounded) |
| black | 1/0 | 84.3% | 7.4% | 82.7% | −76.9 | −1.6 |
| hispanic | 1/0 | 5.9% | 7.2% | 4.3% | 1.3 | −1.6 |
| married | 1/0 | 18.9% | 71.2% | 15.1% | 52.3 | −3.8 |
| no degree | 1/0 | 70.8% | 29.6% | 70.8% | −41.2 | 0.0 (exact count) |
| 1974 earnings | number | 2,095.6 | 14,016.8 | 2,021.9 | 11,921.2 | −73.7 |
| 1975 earnings | number | 1,532.1 | 13,650.8 | 1,310.1 | 12,118.7 | −222.0 |

- Hispanic: the difference grew from 1.3 to (−1.6) points. Matching made it bigger, though it's still under 2 points.
- Every other difference shrank.

It also fails the size test, on one column, age:

| column | type | difference after matching | spread among the treated men (185 men) | size = difference ÷ spread | passes (size under 0.1)? |
|---|---|---|---|---|---|
| age | number | −2.8 | 7.2 | −0.389 | no |
| years of education | number | 0.0 (rounded) | 2.0 | 0.000 (rounded) | yes |
| black | 1/0 | −1.6 | 36.5 | −0.044 | yes |
| hispanic | 1/0 | −1.6 | 23.7 | −0.068 | yes |
| married | 1/0 | −3.8 | 39.3 | −0.097 | yes |
| no degree | 1/0 | 0.0 (exact count) | 45.6 | 0.000 (exact count) | yes |
| 1974 earnings | number | −73.7 | 4,886.6 | −0.015 | yes |
| 1975 earnings | number | −222.0 | 3,219.3 | −0.069 | yes |

- The matches are 2.8 years younger than the treated men on average.
- Hispanic grew, but its size, −0.068, still passes.

## Interview angle

> "We matched users who used feature X to similar users who didn't, using a propensity score. The matched users with X spend 10% more. Is that the effect of X?"

A strong answer: not on its own. It's the effect of X only if all three hold:
- Balance: after matching, the users with X and their matches are alike on every column used, by the size test. If not, part of the 10% extra spending may come from those differences, not from X.
- Overlap: the users with X have users without X with a similar score. If not, those users get matches unlike them. And if a few users without X are reused for many matches, the estimate gets shakier.
- No confounder without a column. No data can prove this one. Matching only makes the users alike on the columns it sees. How keen a user is, beyond what the columns measure, is the usual trait with no column: keen users try more features and also spend more anyway.

## Key takeaways

- Matching estimates the effect on the treated men by comparing each one with the comparison man most like him before the program.
- With many columns, exact matches are rare: only 12 of the 185 treated men had a PSID man the same on all 8 columns. The propensity score, each man's chance of being a treated man given his columns, turns the columns into one number to match on.
- The balance check shows whether matching worked. Every difference shrinking is an easy test but a weak one. The size test, (difference after matching) ÷ (spread among the treated men) under 0.1, is the common choice. With the PSID men, matching passed the easy test and failed the size test on 4 columns.
- On the real data, matching moved the naive gap, (−$15,204.8), to $2,125.7, against a benchmark of $1,794.3. Its 95% confidence interval, $264.5 to $3,986.9, doesn't contain 0.
- Overlap matters: 23 treated men had no PSID man with a score as high as theirs, and one PSID man was the match of 26 treated men.
- Leaving out the comparison men unlike any treated man moved regression adjustment's estimate from $790.5 to $1,931.2: as close to the benchmark as matching, as far as this data can tell.
- What matching can't fix: a confounder with no column, such as motivation.